In [1]:
import pandas as pd

questions = pd.read_csv('data/processed/questions.csv')
pairs = pd.read_csv('data/processed/pairs.csv')
answers = pd.read_csv('data/processed/answers.csv')

print(questions.shape)
print(pairs.shape)
print(answers.shape)

(6107, 8)
(944, 3)
(9016, 4)


In [ ]:
# duplicate pairs vs linked pairs?
print(pairs['LinkTypeId'].value_counts())

LinkTypeId
1    777
3    167
Name: count, dtype: int64


In [14]:
# explore questions
print(questions.columns.tolist())
print()
print(questions.iloc[1])

['Id', 'Title', 'Body', 'Tags', 'Score', 'AcceptedAnswerId', 'AnswerCount', 'CreationDate', 'title_length']

Id                                                                  3
Title                          How is offside determined in football?
Body                The offside rule on the pitch is often unclear...
Tags                                     |football|rules|officiating|
Score                                                              55
AcceptedAnswerId                                                 14.0
AnswerCount                                                         7
CreationDate                                  2012-02-08T20:11:02.963
title_length                                                       38
Name: 1, dtype: object


In [19]:
# Title length distribution
questions['title_length'] = questions['Title'].str.len()
print(questions['title_length'].describe())

count    6107.000000
mean       58.603570
std        22.937957
min        15.000000
25%        42.000000
50%        56.000000
75%        72.000000
max       150.000000
Name: title_length, dtype: float64


In [21]:
# What sports are in this corpus?
from collections import Counter

all_tags = []
for tag_string in questions['Tags'].dropna():
    tags = tag_string.strip('|').split('|')
    all_tags.extend(tags)

tag_counts = Counter(all_tags)
for tag, count in tag_counts.most_common(20):
    print(f"{tag}: {count}")

rules: 1783
football: 1105
cricket: 829
baseball: 596
trivia: 567
american-football: 541
basketball: 409
tennis: 389
officiating: 384
nfl: 370
statistics: 355
equipment: 333
history: 287
mlb: 263
terminology: 262
nba: 246
ice-hockey: 219
technique: 215
olympics: 168
formula-1: 167


In [6]:
# How many questions are tagged football?
football_questions = questions[questions['Tags'].str.contains('football', na=False)]
print(f"Football questions: {len(football_questions)}")

# Show 5 example football questions
print(football_questions['Title'].head(10).tolist())

Football questions: 1656
['How is offside determined in football?', 'As a defensive lineman, can I intentionally encroach to stop the clock?', 'What seat is the best overall seat in an American football stadium?', 'Where did the term "soccer" originate?', 'Why would you ever allow an opponent to score?', 'Difference between one gap and two gap techniques for defensive linemen', 'How is the BCS rank determined?', 'How do defenders score goals when part of a formation', 'Why are substitutions commonly made right at the end of the game?', 'Does icing a fieldgoal kicker actually work?']


In [7]:
# Separate association football from american football
assoc_football = questions[
    questions['Tags'].str.contains('association-football', na=False) |
    (questions['Tags'].str.contains('football', na=False) & 
     ~questions['Tags'].str.contains('american-football|nfl', na=False))
]

american_football = questions[
    questions['Tags'].str.contains('american-football|nfl', na=False)
]

print(f"Association football (soccer): {len(assoc_football)}")
print(f"American football: {len(american_football)}")

Association football (soccer): 1115
American football: 620


## Data Quality Note
The `football` tag on Sports SE is ambiguous — it includes both association 
football (soccer) and American football. True association football questions: 
~1,115. We'll use `association-football` tag or exclude `american-football/nfl` 
tags when evaluating football-specific recall in Week 4.

In [8]:
# Show 5 example duplicate pairs
duplicates = pairs[pairs['LinkTypeId'] == 3].head(5)

for _, row in duplicates.iterrows():
    q1 = questions[questions['Id'] == row['PostId']]['Title'].values
    q2 = questions[questions['Id'] == row['RelatedPostId']]['Title'].values
    if len(q1) and len(q2):
        print(f"Q1: {q1[0]}")
        print(f"Q2: {q2[0]}")
        print()

Q1: Does the run count, or is it considered a force-out
Q2: Does a run score if the batter is tagged for the 3rd out before he reaches first?

Q1: Player with 350 or more goals in domestic league
Q2: Players with 300 or more goals for a single team in top flight domestic league (European clubs)

Q1: age cut off date for cwc u19
Q2: How are the age restrictions followed by ICC for under 19 cricket tournaments?

Q1: What counts as a deliberate save for offside?
Q2: Is there offside when the ball deflects off an opponent?

Q1: Is it possible to win a cricket match by 0 wickets?
Q2: Do the laws of cricket allow running on the last ball, even after winning?



## Pair Quality Observation
Duplicate pairs (LinkTypeId=3) vary in quality. Some are genuine 
rewordings of the same question. Others are loosely related questions 
that a community moderator linked. This noise in labels will limit 
fine-tuning gains — a known limitation worth mentioning in the writeup.

In [22]:
# Show 5 example linked pairs (LinkTypeId=1)
linked = pairs[pairs['LinkTypeId'] == 1].head(5)

for _, row in linked.iterrows():
    q1 = questions[questions['Id'] == row['PostId']]['Title'].values
    q2 = questions[questions['Id'] == row['RelatedPostId']]['Title'].values
    if len(q1) and len(q2):
        print(f"Q1: {q1[0]}")
        print(f"Q2: {q2[0]}")
        print()

Q1: Does the run count, or is it considered a force-out
Q2: Does a run score if the batter is tagged for the 3rd out before he reaches first?

Q1: Player with 350 or more goals in domestic league
Q2: Players with 300 or more goals for a single team in top flight domestic league (European clubs)

Q1: Finland goal song at IIHF WM 2019
Q2: What is the walk out song World Cup 2018?

Q1: age cut off date for cwc u19
Q2: How are the age restrictions followed by ICC for under 19 cricket tournaments?

Q1: How many Premier League teams will play in the 17-18 Champions League?
Q2: What will champions league authority do if the champions don't qualify for the next season's tournament?



## Linked vs Duplicate Pair Quality
Linked pairs (LinkTypeId=1) are significantly noisier than duplicates 
(LinkTypeId=3). May be worth experimenting with training on duplicates 
only (167 pairs) vs all pairs (944) to see which gives better recall. 
Trade-off: cleaner signal vs more data.'''

In [10]:
print(answers['Score'].describe())
print(f"\nQuestions with accepted answers: {questions['AcceptedAnswerId'].notna().sum()}")
print(f"Questions with zero answers: {(questions['AnswerCount'] == 0).sum()}")

count    9016.000000
mean        3.464729
std         4.375837
min       -10.000000
25%         1.000000
50%         2.000000
75%         5.000000
max        82.000000
Name: Score, dtype: float64

Questions with accepted answers: 3291
Questions with zero answers: 456


In [11]:
# Any empty titles?
print(f"Empty titles: {questions['Title'].isna().sum()}")

# Any very short titles?
print(f"Titles under 10 chars: {(questions['title_length'] < 10).sum()}")

# Any duplicate question titles?
print(f"Duplicate titles: {questions['Title'].duplicated().sum()}")

Empty titles: 0
Titles under 10 chars: 0
Duplicate titles: 1


In [12]:
# Find the duplicate title
dup_title = questions[questions['Title'].duplicated(keep=False)]
print(dup_title[['Id', 'Title', 'Tags']])

     Id                                         Title      Tags
38  101  How can I improve my endurance while skiing?  |skiing|
39  102  How can I improve my endurance while skiing?  |skiing|


In [13]:
print("=== Dataset Summary ===")
print(f"Total questions: {len(questions)}")
print(f"Total answers: {len(answers)}")
print(f"Total pairs: {len(pairs)}")
print(f"  - Duplicate pairs: {len(pairs[pairs['LinkTypeId']==3])}")
print(f"  - Linked pairs: {len(pairs[pairs['LinkTypeId']==1])}")
print(f"Association football questions: 1115")
print(f"American football questions: 620")
print(f"Questions with accepted answers: {questions['AcceptedAnswerId'].notna().sum()}")
print(f"Questions with zero answers: {(questions['AnswerCount']==0).sum()}")
print(f"Avg title length: {questions['title_length'].mean():.1f} chars")

=== Dataset Summary ===
Total questions: 6107
Total answers: 9016
Total pairs: 944
  - Duplicate pairs: 167
  - Linked pairs: 777
Association football questions: 1115
American football questions: 620
Questions with accepted answers: 3291
Questions with zero answers: 456
Avg title length: 58.6 chars
